<!-- NB_DOC_INTRO_V1 -->
# AutoML — Automated Machine Learning

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**AutoML** = automatiser le pipeline ML : choix d'algorithme, **hyperparametres**, parfois meme feature engineering et stacking. Le but : obtenir un modele competitif **sans expertise ML** et **avec un budget temps fixe**.

Ce notebook couvre les **5 outils Python majeurs** d'AutoML tabulaire en 2024-2025, avec code execute pour chacun (sur dataset `breast_cancer` sklearn, 569 lignes, 30 features, binaire) :

1. **FLAML** (Microsoft Research) — algo CFO/BlendSearch, le plus efficient sur petit budget
2. **AutoGluon** (AWS) — stacking et bagging massifs, top sur Kaggle tabular
3. **auto-sklearn** (Freiburg) — meta-learning + Bayesian (Linux only)
4. **PyCaret** — wrapper "low-code" autour de sklearn + voting
5. **H2O AutoML** — JVM-based, scale gros volume

**Quand utiliser AutoML vs custom ?**
- ✅ **POC rapide** : FLAML 60s donne souvent du `>= 90%` du best score humain
- ✅ **Bench equitable** de plusieurs algos
- ✅ **Pas d'expert ML** dispo
- ❌ **Production critique** : audite ce qu'il fait (interpretabilite, fairness)
- ❌ **Donnees specifiques** (TS, NLP, image) ou approches custom dominent (preferer Optuna cible)
- ❌ **Donnees < 1000 lignes** : tester en CV manuelle, le hasard domine
- ❌ **Si Optuna sur XGBoost suffit** : pas besoin du surdimensionne

> 🎯 **Regle d'or** : AutoML = **point de depart**, pas point d'arrivee. Toujours auditer le pipeline retourne (features utilisees, type de modele, calibration, biais).

## Plan

1. Installation et donnees jouet (sklearn breast_cancer)
2. **FLAML** — execution + lecture du best config + feature importance
3. **AutoGluon** (decommenter — lourd a installer)
4. **PyCaret** (decommenter — lourd)
5. **auto-sklearn** (decommenter — Linux only)
6. **Tableau comparatif** : algos couverts, vitesse, RAM, qualite typique
7. **Optuna cible** — alternative quand un seul modele suffit
8. Audit post-AutoML : SHAP, calibration, fairness
9. Pieges courants
10. References


## 1. Installation et donnees jouet

On utilise le **Breast Cancer Wisconsin** dataset (569 samples, 30 features, binaire malignant/benign). Petit pour rester rapide en CPU.


In [ ]:
# pip install -q flaml scikit-learn numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, classification_report,
)

SEED = 42

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)

print(f"Train : {X_train.shape}, Test : {X_test.shape}")
print(f"Class balance train : {y_train.value_counts(normalize=True).to_dict()}")
print(f"Class balance test  : {y_test.value_counts(normalize=True).to_dict()}")
print(f"\nFeatures (5 premieres) : {list(X.columns[:5])}")

## 2. FLAML — execution

**FLAML** (Fast and Lightweight AutoML) utilise l'algo **CFO** (Cost-Frugal Optimization) qui explore intelligemment l'espace des HP a budget contraint. Avantages : rapide, peu de RAM, ecosystem sklearn-friendly.

API : `automl.fit(X, y, task=..., time_budget=..., estimator_list=...)`. Le critere est `metric` (string standard sklearn).


In [ ]:
from flaml import AutoML

automl = AutoML()
settings = {
    "time_budget": 60,                                              # secondes
    "metric": "roc_auc",                                            # ou f1, accuracy, log_loss...
    "task": "classification",
    "estimator_list": ["lgbm", "xgboost", "rf", "extra_tree", "lrl1"],  # restrict pour rapide
    "eval_method": "cv",
    "n_splits": 5,
    "seed": SEED,
    "verbose": 0,
    "log_file_name": "",                                           # disable file log
}

automl.fit(X_train=X_train, y_train=y_train, **settings)
print(f"\nMeilleur estimateur : {automl.best_estimator}")
print(f"Meilleurs hyperparams : {automl.best_config}")

**Lecture du resultat** : FLAML retourne le meilleur **estimateur** et la **config** correspondante. On peut maintenant predire normalement, et auditer les choix.


In [ ]:
# Predictions
y_pred = automl.predict(X_test)
y_proba = automl.predict_proba(X_test)[:, 1]

# Eval sur test
print(f"Test accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Test F1       : {f1_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["malignant", "benign"], digits=3))

### 2.1 Feature importance du meilleur modele

FLAML expose le **modele sous-jacent** via `automl.model.estimator`. On peut donc utiliser ses methodes natives.


In [ ]:
# Le modele sous-jacent (sklearn-like)
underlying = automl.model.estimator
print(f"Type : {type(underlying).__name__}")

# Feature importance (si disponible)
if hasattr(underlying, "feature_importances_"):
    imps = underlying.feature_importances_
    feat_imp = pd.DataFrame({
        "feature":    X.columns,
        "importance": imps,
    }).sort_values("importance", ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(feat_imp["feature"][::-1], feat_imp["importance"][::-1])
    ax.set_title(f"Top 10 features (importance) - {type(underlying).__name__}")
    plt.tight_layout()
    plt.show()
else:
    print("Pas de feature_importances_ pour ce modele")

### 2.2 Historique des essais FLAML

FLAML log chaque essai dans un dict. Permet d'analyser quels estimateurs ont ete essayes et comment les scores ont evolue.


In [ ]:
# Pour une analyse fine de l'historique, lire le log file
# (ici on l'a desactive avec log_file_name="")
# Avec log_file_name="flaml.log", on aurait :
#   from flaml.data import get_output_from_log
#   time_history, best_valid_loss_history, valid_loss_history, config_history, metric_history = \
#       get_output_from_log(filename="flaml.log", time_budget=60)

# Affichage rapide du score final
print(f"Best validation score : {1 - automl.best_loss:.4f}  (1 - loss)")
print(f"Best estimator        : {automl.best_estimator}")
print(f"Time taken            : {automl.time_to_find_best_model:.1f}s")

## 3. AutoGluon — code de reference (decommenter pour executer)

AutoGluon est plus lourd a installer (~ 1.5 GB) et plus lent que FLAML. Il fait du **stacking** automatique sur 5-15 modeles, generalement champion sur Kaggle tabular.

> ⚠️ **Installation** : `pip install -q autogluon.tabular[all]`. Sur Mac M1 : verifier compatibilite torch.


In [ ]:
# # pip install -q autogluon.tabular[all]
# from autogluon.tabular import TabularDataset, TabularPredictor
# import pandas as pd
#
# # AutoGluon attend un DataFrame avec target inclus
# train_df = X_train.copy()
# train_df["target"] = y_train
# test_df = X_test.copy()
# test_df["target"] = y_test
#
# predictor = TabularPredictor(label="target", eval_metric="roc_auc", verbosity=2).fit(
#     train_df,
#     time_limit=120,                  # 2 minutes
#     presets="medium_quality",         # ou : "best_quality" (longer), "good_quality_faster"
#     auto_stack=True,                  # active le stacking
# )
#
# # Eval sur test
# perf = predictor.evaluate(test_df)
# print(perf)
#
# # Leaderboard : tous les modeles entraines + scores
# lb = predictor.leaderboard(test_df, silent=True)
# print(lb)
print("AutoGluon : decommenter le code ci-dessus (necessite installation autogluon.tabular)")

## 4. PyCaret — code de reference (decommenter pour executer)

**PyCaret** est un wrapper "low-code" qui simplifie sklearn. API similaire a un notebook Kaggle.

> ⚠️ **Installation** : `pip install -q pycaret`. Tres opinionated (s'attend a un DataFrame avec target), parfois en conflit avec d'autres libs.


In [ ]:
# # pip install -q pycaret
# from pycaret.classification import (
#     setup, compare_models, tune_model, finalize_model, predict_model, plot_model,
# )
#
# # Prep
# data_df = X_train.copy()
# data_df["target"] = y_train
#
# # 1. Setup
# s = setup(
#     data=data_df,
#     target="target",
#     session_id=SEED,
#     train_size=0.8,
#     verbose=False,
# )
#
# # 2. Comparer TOUS les modeles
# best = compare_models(n_select=3, sort="AUC")    # top 3 par AUC
#
# # 3. Tune le meilleur
# tuned = tune_model(best[0])
#
# # 4. Finalize (train sur tout le data)
# final = finalize_model(tuned)
#
# # 5. Predict
# preds = predict_model(final, data=X_test)
# print(preds.head())
print("PyCaret : decommenter (necessite installation pycaret)")

## 5. auto-sklearn — code de reference (Linux only)

**auto-sklearn** (Freiburg) est la **reference academique** de l'AutoML (Feurer et al. 2015, 2020). Basee sur **SMAC** (Sequential Model-based Algorithm Configuration) + **meta-learning** (utilise des connaissances acquises sur d'autres datasets pour demarrer).

> ⚠️ **Linux only** generalement (depend de SMAC C++). Pour Windows / Mac : utiliser FLAML.


In [ ]:
# # pip install -q auto-sklearn      # Linux uniquement
# import autosklearn.classification
#
# automl_sk = autosklearn.classification.AutoSklearnClassifier(
#     time_left_for_this_task=120,
#     per_run_time_limit=30,
#     ensemble_size=10,
#     n_jobs=-1,
#     seed=SEED,
# )
# automl_sk.fit(X_train, y_train)
#
# # Statistiques
# print(automl_sk.sprint_statistics())
#
# # Eval
# print(f"Test accuracy : {automl_sk.score(X_test, y_test):.4f}")
#
# # Composition de l'ensemble
# print(automl_sk.leaderboard())
print("auto-sklearn : Linux uniquement, decommenter sur env compatible")

## 6. Tableau comparatif des outils AutoML

| Outil | Backend | Algos couverts | OS | Forces | Limites |
|---|---|---|---|---|---|
| **FLAML** | sklearn + XGB + LGB + CatBoost | ~ 10 | Tout | Tres rapide (CFO), peu de RAM, sklearn-friendly | Tabulaire only |
| **AutoGluon** | sklearn + XGB + LGB + CatBoost + NN | ~ 15 + stacking | Tout | Top Kaggle, multi-modalite (image/text) | Lourd (~ 1.5 GB), lent |
| **auto-sklearn** | sklearn only | ~ 15 | Linux | Reference academique, meta-learning | Lent, sklearn-only, OS-restrict |
| **PyCaret** | wrapper sklearn | ~ 25 | Tout | Low-code, user-friendly | Moins flexible, opinionated |
| **H2O AutoML** | H2O JVM | ~ 20 + stacking | Tout (JVM) | Scale gros volume, prod ready | API Java-flavored, JVM |
| **TPOT** | sklearn + GP | ~ 20 | Tout | Decouverte de pipelines via genetic | Lent, peu maintenu |

### Quel outil pour quel cas ?

| Cas | Recommandation |
|---|---|
| **POC rapide (1-2 min)** | **FLAML** |
| **Comparaison equitable plusieurs algos** | **FLAML** ou **PyCaret** |
| **Top score sur Kaggle tabular** | **AutoGluon** (`best_quality`) |
| **Tabular + image + text** | **AutoGluon Multimodal** |
| **Recherche academique / reproductibilite** | **auto-sklearn** |
| **Equipe sans expert ML** | **PyCaret** (low-code) ou **AutoGluon** |
| **Scale enterprise / Spark** | **H2O AutoML** |
| **Budget GPU dispo, vouloir le top score** | **AutoGluon best_quality + GPU** |


## 7. Optuna cible — alternative quand un seul modele suffit

**AutoML** cherche **modele + HP**. **Optuna** cherche les **HP d'un modele DONNE**. Si tu sais que XGBoost est l'outil, ne lance pas AutoML, lance Optuna cible.


In [ ]:
# pip install -q optuna xgboost
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 50, 500),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "learning_rate":     trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
    }
    model = XGBClassifier(**params, random_state=SEED, eval_metric="logloss", n_jobs=-1)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1).mean()
    return score

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f"Best trial AUC : {study.best_value:.4f}")
print(f"Best params    : {study.best_params}")

In [ ]:
# Fit le best sur tout le train et eval sur test
best_xgb = XGBClassifier(**study.best_params, random_state=SEED, eval_metric="logloss", n_jobs=-1)
best_xgb.fit(X_train, y_train)
y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:, 1]
print(f"Test AUC (Optuna)    : {roc_auc_score(y_test, y_proba):.4f}")
print(f"Test AUC (FLAML)     : (calcule plus haut)")

## 8. Audit post-AutoML — toujours faire avant la prod

AutoML est une boite (semi) noire. Avant deploiement :

| Check | Outil | Section dans la collection |
|---|---|---|
| **Performance par segment** (genre, age, region) | groupby + accuracy/f1 par groupe | [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) |
| **SHAP / LIME** sur le modele final | `shap`, `lime` | [ML_Explication_Feature_Importance_Selection.ipynb](./ML_Explication_Feature_Importance_Selection.ipynb) |
| **Calibration** | `calibration_curve`, `brier_score_loss` | [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) |
| **Fairness** | `fairlearn`, `aif360` | [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) |
| **Drift potentiel** | `evidently`, `nannyml` | [MLOps_Drift_Monitoring.ipynb](./MLOps_Drift_Monitoring.ipynb) |
| **Temps d'inference** | `timeit`, profiling | a verifier pour SLA |
| **Robustesse** (perturbations features) | `eli5.show_weights`, adversarial | papier specifique |

### Calibration rapide du modele FLAML


In [ ]:
from sklearn.calibration import calibration_curve

# Reliability diagram
frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfectly calibrated")
ax.plot(mean_pred, frac_pos, "o-", label="FLAML best model")
ax.set(xlabel="Mean predicted probability", ylabel="Fraction of positives",
       title="Reliability diagram (FLAML best model)")
ax.legend()
plt.show()

from sklearn.metrics import brier_score_loss
print(f"Brier score (plus bas = mieux) : {brier_score_loss(y_test, y_proba):.4f}")

## 9. Pieges courants

| ❌ Piege | ✅ Correction |
|---|---|
| Lancer AutoML, reporter le score, deployer | Auditer (SHAP, calibration, fairness) AVANT prod |
| AutoML sur < 1000 lignes | Trop variable, faire CV manuelle avec un seul modele |
| `time_budget=60` pour un probleme dur | Adapter au probleme (10 min - 1h ; sur Kaggle souvent > 4h) |
| `metric="accuracy"` sur desequilibre | F1, AUC, MCC (cf. [DS_Metrics_Reference](./DS_Metrics_Reference.ipynb)) |
| AutoML sur donnees temporelles avec KFold random | TimeSeriesSplit ou walk-forward (eval_method customisee) |
| Refit avec les memes seeds, esperer reproductibilite parfaite | Seeds ne suffisent pas toujours (n_jobs nondeterminisme) |
| Comparer FLAML 60s vs AutoGluon 4h | Comparer a budget egal |
| Ignorer le scaling cout-business | Reporter aussi le cout pondere (cf. metriques business) |
| Stack tres profond pour 0.1% de gain | Verifier que la complexite est justifiee (temps d'inference, maintenance) |
| Pas de baseline | TOUJOURS comparer au modele naif (majoritaire / mean / lag1 pour TS) |

### Baseline naive a TOUJOURS reporter


In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
y_dummy_pred = dummy.predict(X_test)
y_dummy_proba = dummy.predict_proba(X_test)[:, 1]

print(f"Baseline (majority) : acc={accuracy_score(y_test, y_dummy_pred):.4f}, "
      f"AUC={roc_auc_score(y_test, y_dummy_proba):.4f}")
print(f"FLAML best          : acc={accuracy_score(y_test, automl.predict(X_test)):.4f}, "
      f"AUC={roc_auc_score(y_test, automl.predict_proba(X_test)[:, 1]):.4f}")

## 10. References

### Documentation officielle
- **FLAML** : https://microsoft.github.io/FLAML/
- **AutoGluon** : https://auto.gluon.ai/
- **auto-sklearn** : https://automl.github.io/auto-sklearn/
- **PyCaret** : https://pycaret.org/
- **H2O AutoML** : https://docs.h2o.ai/h2o/latest-stable/h2o-docs/automl.html
- **Optuna** : https://optuna.readthedocs.io/

### Papers fondateurs
- Feurer et al. (2015). *Efficient and Robust Automated Machine Learning* (auto-sklearn).
- Feurer et al. (2020). *Auto-Sklearn 2.0: Hands-free AutoML via Meta-Learning*.
- Wang et al. (2021). *FLAML: A Fast and Lightweight AutoML Library*.
- Erickson et al. (2020). *AutoGluon-Tabular: Robust and Accurate AutoML for Structured Data*.
- Akiba et al. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework*.

### Bench publics
- AutoML benchmark (OpenML) : https://github.com/openml/automlbenchmark
- Kaggle Tabular Playground series (souvent dominees par AutoGluon)

### Voir aussi (collection)
- [ML_Regression_Classification_CV_GridSearch.ipynb](./ML_Regression_Classification_CV_GridSearch.ipynb) — bench manuel
- [ML_Optimisation_de_Modèles.ipynb](./ML_Optimisation_de_Modèles.ipynb) — Optuna cible
- [ML_Explication_Feature_Importance_Selection.ipynb](./ML_Explication_Feature_Importance_Selection.ipynb) — audit SHAP/LIME
- [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) — choix de metriques
